# Assignment Title: NLP using Python-II

**Feature extraction from text, sentiment analysis and text generation**

## Q1. Text preprocessing

Write a unique paragraph about a favorite topic.

In [ ]:
import re
import nltk
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

paragraph = "Technology has changed the way people learn, communicate, and solve problems. Smartphones help us stay connected with friends and family across long distances. Online courses make education available to people who cannot attend a classroom every day. Artificial intelligence can help people analyze information and complete repetitive tasks. However, technology should be used responsibly because privacy and screen time also matter. I think technology is most useful when it supports human creativity and makes everyday life easier."

lower_text = paragraph.lower()
clean_text = re.sub(r'[^a-z\s]', '', lower_text)
words = word_tokenize(clean_text)
sentences = sent_tokenize(paragraph)
split_words = clean_text.split()
nltk_words = word_tokenize(clean_text)
stop_words = set(stopwords.words('english'))
filtered_words = [word for word in words if word not in stop_words]

print("Lowercase text:\n", lower_text)
print("\nText without punctuation:\n", clean_text)
print("\nSentences:", sentences)
print("\nPython split():", split_words)
print("\nNLTK word_tokenize():", nltk_words)
print("\nWords after stopword removal:\n", filtered_words)

fdist = FreqDist(filtered_words)
fdist.plot(15, title="Word Frequency Distribution")
plt.show()

## Q2. Stemming and lemmatization

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet')
nltk.download('omw-1.4')
alphabetic_words = re.findall(r'\b[A-Za-z]+\b', paragraph.lower())
words_without_stopwords = [word for word in alphabetic_words if word not in stop_words]

porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = [porter.stem(word) for word in words_without_stopwords]
lemmatized_words = [lemmatizer.lemmatize(word) for word in words_without_stopwords]

print("Alphabetic words:", alphabetic_words)
print("Words after stopword removal:", words_without_stopwords)
print("Porter stemming:", stemmed_words)
print("WordNet lemmatization:", lemmatized_words)

Stemming cuts word endings using rules, so it can produce word fragments. Lemmatization uses a vocabulary and linguistic analysis to return a dictionary form. Lemmatization is preferable when readable words and meaning matter; stemming is useful when a quick reduction in word forms is sufficient.

## Q3. Bag of Words and TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

texts = [
    "Artificial intelligence helps analyze data and automate tasks.",
    "Blockchain stores records securely across a distributed network.",
    "Technology tools help people learn and solve everyday problems."
]

count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(texts)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=count_vectorizer.get_feature_names_out(), index=["Text 1", "Text 2", "Text 3"])
print("Bag of Words:")
display(bow_df)

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out(), index=["Text 1", "Text 2", "Text 3"])
print("TF-IDF scores:")
display(tfidf_df)

feature_names = tfidf_vectorizer.get_feature_names_out()
for i, row in enumerate(tfidf_matrix.toarray()):
    top_indices = row.argsort()[::-1][:3]
    print(f"Text {i + 1} top keywords:")
    for idx in top_indices:
        print(feature_names[idx], round(row[idx], 3))

## Q4. Text similarity

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

text1 = "Artificial intelligence helps computers learn patterns and make predictions from data. It is used in healthcare, education, and business. Models can automate repetitive work and support decision-making."
text2 = "Blockchain is a distributed ledger that stores records in linked blocks. It can improve transparency and make records difficult to change. It is used in digital transactions and supply chains."

def preprocess(text):
    text = re.sub(r'[^a-z\s]', ' ', text.lower())
    return word_tokenize(text)

tokens1 = preprocess(text1)
tokens2 = preprocess(text2)
set1 = set(tokens1)
set2 = set(tokens2)
jaccard = len(set1 & set2) / len(set1 | set2)

vectorizer = TfidfVectorizer()
tfidf_vectors = vectorizer.fit_transform([text1, text2])
cosine = cosine_similarity(tfidf_vectors[0:1], tfidf_vectors[1:2])[0][0]

print("Text 1 tokens:", tokens1)
print("Text 2 tokens:", tokens2)
print("Jaccard similarity:", round(jaccard, 4))
print("Cosine similarity:", round(cosine, 4))
print("Jaccard compares shared unique words, while TF-IDF cosine similarity compares weighted word vectors. Here, cosine similarity can give a more useful comparison because it accounts for how informative words are across both texts.")

## Q5. Product review and sentiment analysis

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

nltk.download('vader_lexicon')

reviews = [
    "The service is excellent and the product is easy to use.",
    "The app is useful, fast, and reliable.",
    "The product is disappointing and the support is poor.",
    "The delivery was slow and the packaging was damaged.",
    "The product works well, although the price is high."
]

sia = SentimentIntensityAnalyzer()
positive_reviews = []

for i, review in enumerate(reviews, start=1):
    scores = sia.polarity_scores(review)
    compound = scores['compound']
    label = "Positive" if compound >= 0.05 else "Negative" if compound <= -0.05 else "Neutral"
    print(f"Review {i}: {label} | polarity={compound:.3f} | subjectivity not calculated by VADER" )
    if label == "Positive":
        positive_reviews.append(review)

if positive_reviews:
    cloud = WordCloud(width=800, height=400, background_color="white").generate(" ".join(positive_reviews))
    plt.figure(figsize=(10, 5))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.show()

In [ ]:
from textblob import TextBlob
from wordcloud import WordCloud

reviews = [
    "The service is excellent and the product is easy to use.",
    "The app is useful, fast, and reliable.",
    "The product is disappointing and the support is poor.",
    "The delivery was slow and the packaging was damaged.",
    "The product works well, although the price is high."
]

positive_reviews = []

for i, review in enumerate(reviews, start=1):
    analysis = TextBlob(review).sentiment
    polarity = analysis.polarity
    subjectivity = analysis.subjectivity
    label = "Positive" if polarity > 0 else "Negative" if polarity < 0 else "Neutral"
    print(f"Review {i}: {label} | polarity={polarity:.3f} | subjectivity={subjectivity:.3f}")
    if label == "Positive":
        positive_reviews.append(review)

if positive_reviews:
    cloud = WordCloud(width=800, height=400, background_color="white").generate(" ".join(positive_reviews))
    plt.figure(figsize=(10, 5))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.show()

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

training_text = """Technology helps people learn new skills and share ideas. People use digital tools to communicate, solve problems, and create useful things. Learning takes time and practice. Good technology can support creativity, improve access to knowledge, and make everyday tasks easier. People should use technology carefully and protect their privacy."""

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
sequences = tokenizer.texts_to_sequences([training_text])[0]
vocab_size = len(tokenizer.word_index) + 1

input_sequences = []
for i in range(1, len(sequences)):
    input_sequences.append(sequences[:i + 1])

max_length = max(len(seq) for seq in input_sequences)
padded_sequences = np.zeros((len(input_sequences), max_length), dtype=int)
for i, seq in enumerate(input_sequences):
    padded_sequences[i, -len(seq):] = seq

X = padded_sequences[:, :-1]
y = to_categorical(padded_sequences[:, -1], num_classes=vocab_size)

model = Sequential([
    Embedding(vocab_size, 16, input_length=max_length - 1),
    LSTM(32),
    Dense(vocab_size, activation="softmax")
])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(X, y, epochs=100, verbose=0)

def generate_text(seed, count=8):
    result = seed.lower()
    for _ in range(count):
        seq = tokenizer.texts_to_sequences([result])[0]
        seq = seq[-(max_length - 1):]
        padded = np.zeros((1, max_length - 1), dtype=int)
        if seq:
            padded[0, -len(seq):] = seq
        prediction = model.predict(padded, verbose=0)[0]
        next_id = int(np.argmax(prediction))
        next_word = tokenizer.index_word.get(next_id, "")
        if not next_word:
            break
        result += " " + next_word
    return result

print(generate_text("technology helps", 8))
print(generate_text("people use", 8))